In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import numpy as np
import os
import joblib

In [16]:
expression_data = pd.read_csv('expression_data.csv')
gene_variances = expression_data.var(axis=1)
top_5000_genes = gene_variances.sort_values(ascending=False).head(5000).index
top_variable_expression = expression_data.loc[top_5000_genes].T


In [17]:
top_variable_expression

,28617,2067,16178,19514,13317,4946,6817,11299,3563,17233,...,12390,5430,7380,10170,3325,13581,5078,15377,12707,16601
SRR1785238,29.146046,154.923013,23.303464,8.642984,115.124459,101.279450,16.802742,88.370612,124.981039,48.161003,...,9.102982,8.897489,4.069187,8.790723,4.243193,5.459593,5.375897,9.570451,4.832386,4.819880
SRR1785239,28.875818,152.303056,24.785411,8.953094,112.453515,103.339289,16.234747,88.927141,124.981039,48.611523,...,8.520604,9.074295,4.395271,8.947332,4.073924,4.832386,5.525406,9.554921,4.606042,4.047928
SRR1785240,32.691899,144.620796,32.043544,15.021587,113.788757,63.537999,19.367569,63.834684,55.447715,112.453515,...,8.642984,9.067922,4.687382,9.323257,3.613403,3.894425,7.412093,7.809425,5.748477,4.125240
SRR1785241,32.423348,144.620796,29.364786,12.282399,109.734172,70.272715,19.196845,64.930814,51.413458,111.108496,...,7.189754,8.865995,4.042768,9.847058,3.199338,4.426408,6.457956,8.248265,4.955667,3.636402
SRR1785242,22.142084,174.954663,23.099605,9.422501,108.361017,104.481005,31.424953,75.094138,75.554324,75.977505,...,7.978004,9.248195,3.494238,8.435125,3.744418,5.930834,5.105784,9.602848,3.857780,3.343256
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SRR1785321,120.344336,137.748517,127.399822,78.179074,144.620796,79.170604,95.306190,60.071117,75.554324,81.366430,...,8.435125,8.501240,4.844108,9.487463,3.106234,4.093331,4.898765,8.536265,5.185947,3.130590
SRR1785322,25.900860,88.370612,14.068922,11.608351,112.453515,83.793804,8.402887,55.671101,75.094138,58.044237,...,6.429043,8.024073,5.326215,8.460097,2.992443,4.876213,4.033379,9.649546,5.053424,5.152778
SRR1785323,26.765262,88.927141,14.631208,11.780632,112.453515,81.979447,8.017134,56.134138,79.170604,59.349812,...,7.656067,8.265033,5.349425,8.596235,2.984197,5.101057,4.033972,9.515838,5.179786,4.885028
SRR1785324,32.208170,113.788757,38.089461,14.020673,77.253150,41.791489,12.908763,117.782800,103.339289,51.413458,...,8.071794,9.326643,5.310100,8.578132,5.759617,4.342098,5.718947,9.914781,5.234574,4.545490


In [18]:
target = pd.read_csv('../data/processed/group_annotation.csv', index_col=0)

target

,Group
SRR1785238,T0
SRR1785239,T0
SRR1785240,T0
SRR1785241,T0
SRR1785242,T0
...,...
SRR1785321,T3
SRR1785322,T3
SRR1785323,T3
SRR1785324,T3


In [19]:
from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

X_train, X_test, y_train, y_test = train_test_split(top_variable_expression, target['Group'], 
                                                    test_size=0.2, random_state=0, 
                                                    stratify=target['Group'])


In [20]:
#Logistic Regression Model

log_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('log_reg', LogisticRegression(solver='liblinear'))
])

In [21]:
#hyperparameter tuning for logistic regression
param_grid = {
    'log_reg__C': [0.01, 0.1, 1, 10, 100, 1000],
    'log_reg__penalty': ['l1', 'l2']
}
grid_search_lr = GridSearchCV(log_reg, param_grid, cv=5, 
                              scoring='accuracy')
grid_search_lr.fit(X_train, y_train)

,estimator,Pipeline(step...liblinear'))])
,param_grid,"{'log_reg__C': [0.01, 0.1, ...], 'log_reg__penalty': ['l1', 'l2']}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


In [22]:
grid_search_lr.best_params_


{'log_reg__C': 10, 'log_reg__penalty': 'l2'}

In [23]:
best_model_lr = grid_search_lr.best_estimator_ 
y_test_pred_lr = best_model_lr.predict(X_test)
print(classification_report(y_test, y_test_pred_lr))

              precision    recall  f1-score   support

          T0       1.00      1.00      1.00         9
          T3       1.00      1.00      1.00         9

    accuracy                           1.00        18
   macro avg       1.00      1.00      1.00        18
weighted avg       1.00      1.00      1.00        18



In [24]:
#Random Forest Model

rf = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(random_state=0))
])

In [25]:
rf.get_params()

{'memory': None,
 'steps': [('scaler', StandardScaler()),
  ('rf', RandomForestClassifier(random_state=0))],
 'transform_input': None,
 'verbose': False,
 'scaler': StandardScaler(),
 'rf': RandomForestClassifier(random_state=0),
 'scaler__copy': True,
 'scaler__with_mean': True,
 'scaler__with_std': True,
 'rf__bootstrap': True,
 'rf__ccp_alpha': 0.0,
 'rf__class_weight': None,
 'rf__criterion': 'gini',
 'rf__max_depth': None,
 'rf__max_features': 'sqrt',
 'rf__max_leaf_nodes': None,
 'rf__max_samples': None,
 'rf__min_impurity_decrease': 0.0,
 'rf__min_samples_leaf': 1,
 'rf__min_samples_split': 2,
 'rf__min_weight_fraction_leaf': 0.0,
 'rf__monotonic_cst': None,
 'rf__n_estimators': 100,
 'rf__n_jobs': None,
 'rf__oob_score': False,
 'rf__random_state': 0,
 'rf__verbose': 0,
 'rf__warm_start': False}

In [26]:
#random forest hyperparameter tuning 

param_grid_rf = {
    'rf__n_estimators': [50, 100, 200, 300],
    'rf__max_depth': [None, 3, 5, 10, 15, 20],
    'rf__min_samples_split': [2, 5, 10]
}

grid_search_rf = GridSearchCV(rf, param_grid_rf, cv=5,
                                scoring='accuracy')
grid_search_rf.fit(X_train, y_train)

,estimator,Pipeline(step...om_state=0))])
,param_grid,"{'rf__max_depth': [None, 3, ...], 'rf__min_samples_split': [2, 5, ...], 'rf__n_estimators': [50, 100, ...]}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


In [27]:
grid_search_rf.best_params_

{'rf__max_depth': 3, 'rf__min_samples_split': 5, 'rf__n_estimators': 100}

In [28]:
best_model_rf = grid_search_rf.best_estimator_
y_test_pred_rf = best_model_rf.predict(X_test)
print(classification_report(y_test, y_test_pred_rf))


              precision    recall  f1-score   support

          T0       1.00      0.89      0.94         9
          T3       0.90      1.00      0.95         9

    accuracy                           0.94        18
   macro avg       0.95      0.94      0.94        18
weighted avg       0.95      0.94      0.94        18



In [ ]:
# Load cluster labels from assignment 3
clusters = pd.read_csv('../results/cluster_results_k3.csv').set_index('Sample')
label_col = 'Cluster_k3'

# Align features (samples x features) and labels
X = top_variable_expression.copy()
common = X.index.intersection(clusters.index)
if len(common) == 0:
    raise ValueError("No sample IDs match between top_variable_expression.index and cluster_results_k3.csv Sample column.")
X = X.loc[common]
y = clusters.loc[common, label_col].astype(int)

# sanity check
print("Cluster counts:\n", y.value_counts())

# split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

# Random Forest (multiclass)
rf_pipe = Pipeline([
    ('scaler', StandardScaler()),  # optional for RF
    ('rf', RandomForestClassifier(random_state=0))
])
param_grid_rf = {
    'rf__n_estimators': [100, 200],
    'rf__max_depth': [None, 5, 10],
    'rf__min_samples_split': [2, 5]
}
gs_rf = GridSearchCV(rf_pipe, param_grid_rf, cv=5, scoring='accuracy', n_jobs=-1)
gs_rf.fit(X_train, y_train)
best_rf = gs_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test)

print("Random Forest - classification report:")
print(classification_report(y_test, y_pred_rf))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_rf))

# Save model and predictions
joblib.dump(best_rf, '../results/rf_multiclass_k3_model.joblib')
pd.DataFrame({
    'sample_id': X_test.index,
    'true_cluster': y_test.values,
    'pred_cluster_rf': y_pred_rf
}).to_csv('../results/rf_multiclass_k3_predictions.csv', index=False)

Cluster counts:
 Cluster_k3
0    50
1    30
2     8
Name: count, dtype: int64


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. Use OneVsRestClassifier(LogisticRegression(..)) i

Logistic Regression - classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         6
           2       1.00      1.00      1.00         2

    accuracy                           1.00        18
   macro avg       1.00      1.00      1.00        18
weighted avg       1.00      1.00      1.00        18

Confusion matrix:
[[10  0  0]
 [ 0  6  0]
 [ 0  0  2]]
Random Forest - classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         6
           2       1.00      1.00      1.00         2

    accuracy                           1.00        18
   macro avg       1.00      1.00      1.00        18
weighted avg       1.00      1.00      1.00        18

Confusion matrix:
[[10  0  0]
 [ 0  6  0]
 [ 0  0  2]]
